In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from multimodal_lancedb import *
from utils import *
from ground_truth import *
from judge import *
import pandas as pd
import lancedb
from prompt import *
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
memory = ConversationBufferMemory(memory_key="history", return_messages=True)# memory buffer

/tmp/ipykernel_3714678/3861419305.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="history", return_messages=True)# memory buffer


In [4]:
db = lancedb.connect("./.lancedb_2")
table_audio = db.open_table("music_audio")
audio_embedding_df = table_audio.to_pandas()

audio_embedding_df

,song_name,song_path,audio_vector
0,My Rhapsody Sounds - Short Version A,music/Assaf Ayalon - My Rhapsody Sounds - Shor...,"[-0.029478367, 0.009709472, 0.05185532, 0.0254..."
1,Laid Back - Short Version A,music/The Mind Sweepers - Laid Back - Short Ve...,"[-0.033787563, 0.040226605, 0.004442984, 0.081..."
2,Far Taj,music/ZISO - Far Taj.mp3,"[-0.018721217, 0.035051364, 0.05141652, 0.0356..."
3,The Stones - Short Version,music/Wild Tulip - The Stones - Short Version.mp3,"[-0.008583562, 0.020294761, 0.0050604204, 0.03..."
4,Fixed - Short Version B,music/Swirling Ship - Fixed - Short Version B.mp3,"[0.0031058518, 0.020443501, 0.0060475464, -0.0..."
...,...,...,...
195,Orchestral News Intro,music/Tomasz_Redman - Orchestral News Intro.mp3,"[0.010141604, -0.017981885, 0.014272054, -0.03..."
196,Upbeat Happy Fun Logo,music/puremusic - Upbeat Happy Fun Logo.mp3,"[-0.031311695, -0.03104818, 0.022266045, 0.022..."
197,Happy Birthday In Paris,music/Music-Ideas - Happy Birthday In Paris.mp3,"[-0.049790498, -0.03626891, 0.059174698, -0.00..."
198,Funny Game Loop,music/honey_lemon - Funny Game Loop.wav,"[-0.045391183, -0.033096816, 0.025908915, -0.0..."


In [5]:
table_text = db.open_table("music_text")
text_embedding_df = table_text.to_pandas()

text_embedding_df

,source,song_name,artist,mood,video_theme,genre,instrument,other_tags,bpm,lmm_description,combined_info,text_vector
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Cinematic, Acoustic, Pop, Folk, Children, Corp...","Acoustic Guitar, Keys",,145.0,A positive and uplifting acoustic folk track w...,"Moods: Uplifting, Happy, Carefree, Love, Playf...","[0.00016941165, -0.011131651, -0.004014101, -0..."
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry",Rock,"Electric, Guitar, Acoustic Drums",,78.0,This is a powerful and energetic rock music tr...,"Moods: Powerful, Serious, Angry. Video Themes:...","[-0.0041819224, -0.01964485, -0.021090291, -0...."
2,Artlist,Far Taj,ZISO,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","World, Electronic, Hip Hop","Ethnic, Electronic Drums, Bass",,96.0,A traditional Indian Bhangra track with modern...,"Moods: Uplifting, Powerful, Carefree, Groovy. ...","[-0.011723319, -0.008885955, 0.0040757894, -0...."
3,Artlist,The Stones - Short Version,Wild Tulip,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Cinematic,Piano,,69.0,This piece is a solo piano instrumental with a...,"Moods: Love, Serious, Dramatic, Sad, Hopeful. ...","[0.0012076573, -0.0034555339, -0.0020917628, -..."
4,Artlist,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,121.0,"The music is mysterious and dramatic, featurin...","Moods: Serious, Dramatic, Scary, Dark. Video T...","[-0.0021377725, -0.012590199, -0.013713269, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",corporate,strings,global,125.0,This is a dynamic and uplifting music track th...,"Moods: energetic, epic, powerful, solemn, upli...","[-0.006809455, -0.016746698, -0.016968248, -0...."
196,envato,Upbeat Happy Fun Logo,puremusic,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","acoustic, children","claps, ukulele","melody, youth",NaN,"A positive, upbeat, cheerful, and happy acoust...","Moods: bouncy, bright, catchy, cheerful, energ...","[0.002132031, -0.005020746, -0.0029374287, -0...."
197,envato,Happy Birthday In Paris,Music-Ideas,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","bigband, jazz, retro","accordion, piano, trumpets","france, french, paris",120.0,A fun and lively Latin track featuring a varie...,"Moods: cheerful, funny, happy, lively, playful...","[-0.012678004, -0.011919039, -0.00089095806, -..."
198,envato,Funny Game Loop,honey_lemon,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv","acoustic, children, folk, jazz",bells,loop,170.0,"A casual, jazzy, swing music with vibraphone, ...","Moods: comical, fun, funny, laugh, smile, soft...","[-0.012116052, -0.01674075, 0.010771282, -0.02..."


In [6]:
api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
video_path = 'video/'
input = client.files.upload(file=f"{video_path}18. warm_video.mp4")

In [8]:
# generate summary for image
response = client.models.generate_content(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
                system_instruction=SYS_SUMMARY_PROMPT_VIDEO,
                temperature=0.7
    ),
    contents=[USER_PROMPT_VIDEO, input]
)

video_summary = response.text

print(video_summary)

This heartwarming scene between a grandfather and grandson evokes a sense of nostalgia and familial love, perfect for a sentimental commercial or family-oriented video. Gentle acoustic guitar melodies or a soft piano piece would enhance the tender mood, fitting well within the folk or easy listening genres. The overall atmosphere is optimistic and comforting, suggesting music that is both uplifting and soothing.



In [6]:
# Search the music from query
query = '''Someone is slowly exploring a dark, abandoned castle.'''
results = search_system.search_music(query, top_k=200, use_top_n_context = 1)

df_recommendations = pd.DataFrame(results["final_results"])
df_recommendations

[2025-06-17T00:24:12Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,mood,video_theme,genre,instrument,other_tags,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path,combined_info
0,Otherworldly Tales Logo,Artlist Musical Logos,"Love, Dramatic, Dark, Mysterious","Time-Lapse, Documentary, Landscape, Slow Motio...","Ambient, Cinematic","Piano, Strings, Backing Vocals, Pads",,"A dark, suspenseful, and dramatic track featur...",0.875429,0.967104,0.836139,"audio,text",8.076349e-06,music/Artlist Musical Logos - Otherworldly Tal...,"Moods: Love, Dramatic, Dark, Mysterious. Video..."
1,Talk - Short Version,Alon Ohana,"Serious, Dramatic, Tense, Scary, Dark","Time-Lapse, Drone Shots, Slow Motion, Trailer","Ambient, Cinematic, Classical","Strings, Bells, Synth, Backing Vocals, Keys, Pads",,Mysterious and dark cinematic track with a bea...,0.931673,0.960770,0.919203,"audio,text",7.071895e-06,music/Alon Ohana - Talk - Short Version.mp3,"Moods: Serious, Dramatic, Tense, Scary, Dark. ..."
2,Cyber Warfare,Artlist Musical Logos,"Serious, Dramatic, Tense, Dark","Drone Shots, Landscape, Slow Motion, Shorts, I...","Cinematic, Fantasy","Strings, Bells, Orchestra",,"A dark and intense track with heavy guitars, b...",0.805534,0.776371,0.818032,"audio,text",4.637873e-06,music/Artlist Musical Logos - Cyber Warfare.mp3,"Moods: Serious, Dramatic, Tense, Dark. Video T..."
3,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,"The music is mysterious and dramatic, featurin...",0.860865,1.000000,0.801235,"audio,text",4.125012e-06,music/Swirling Ship - Fixed - Short Version B.mp3,"Moods: Serious, Dramatic, Scary, Dark. Video T..."
4,Black Magic Drama Tension,FlossieWood,"dark, magic, magical, mysterious, mystery, sec...","black magic, film, spy, spying, thriller",drama,"bells, piano, triangle",,"A piano and orchestra play a suspenseful, acti...",0.858096,0.769216,0.896188,"audio,text",3.875091e-06,music/FlossieWood - Black Magic Drama Tension.mp3,"Moods: dark, magic, magical, mysterious, myste..."
5,Creeping Prank,Nuclearmetal,"crazy, creepy, dark, enchanted, fear, fun, ins...","cartoon, comedy, halloween, intro, kids, movie...",horror,"bells, orchestra, strings",loop,"A short, quirky and funny track featuring gloc...",0.832232,0.564864,0.946818,"audio,text",3.640312e-06,music/Nuclearmetal - Creeping Prank.mp3,"Moods: crazy, creepy, dark, enchanted, fear, f..."
6,Happy Halloween 4,Anandavana,"celebrate, cheerful, funny, happy, haunted, my...","cartoon, castle, comedy, halloween, movie, nig...","children, holiday","crotales, harpsichord, pizzicato",,"A fast-paced, playful piece with a mischievous...",0.886676,0.742920,0.948286,"audio,text",3.288749e-06,music/Anandavana - Happy Halloween 4.mp3,"Moods: celebrate, cheerful, funny, happy, haun..."
7,Melancholic Sonata,Artlist Musical Logos,"Love, Dramatic, Sad",Intros & Logos,"Cinematic, Classical",Piano,,The audio is a solo piano performance of a sho...,0.767032,0.826984,0.741338,"audio,text",1.615382e-06,music/Artlist Musical Logos - Melancholic Sona...,"Moods: Love, Dramatic, Sad. Video Themes: Intr..."
8,Horror Trailer,OhmLab,"suspense, scary, freaky, haunting, chilling, a...","zombie, trailer, commercial, movie, thriller, ...","horror, cinematic",,"soundscape, soundtrack",Dark and eerie atmospheric soundscape. Scary h...,0.874584,0.581946,1.000000,"audio,text",1.200450e-06,music/OhmLab - Horror Trailer.mp3,"Moods: suspense, scary, freaky, haunting, chil..."
9,Sad Piano,PineAppleMusic,"agony, deep, depress, emotional, gentle, intim...","autumn, winter","ballad, classical, drama",piano,lyrical,"A subtle, delicate and thoughtful piano piece ...",0.771286,0.938400,0.699666,"audio,text",5.989273e-07,music/PineAppleMusic - Sad Piano.wav,"Moods: agony, deep, depress, emotional, gentle..."


In [7]:
memory.save_context(
        {"input": query},
        {"output": results["explanation"]}
    )

In [8]:
memory_data = memory.load_memory_variables({})
print(memory_data["history"])

[HumanMessage(content='Someone is slowly exploring a dark, abandoned castle.', additional_kwargs={}, response_metadata={}), AIMessage(content='"Otherworldly Tales Logo" is an ideal match for exploring a dark, abandoned castle. Its dark, mysterious, and dramatic mood aligns perfectly with the eerie and suspenseful atmosphere one might experience in such a setting. The use of piano, strings, and synths creates a tense and unsettling ambiance, enhancing the sense of exploration and discovery in a haunted environment. This ambient, cinematic track is well-suited for capturing the slow, suspenseful journey through the castle\'s shadowy corridors. Its suspenseful nature complements the user\'s need for a soundtrack that evokes mystery and intrigue.', additional_kwargs={}, response_metadata={})]


In [9]:
query2 = '''I need a soundtrack that creates a sense of tension.'''
rewrite_query = search_system.rewrite_query_from_history(query2, memory) # rewrite

In [10]:
print(rewrite_query)

Can you recommend a soundtrack that creates a sense of tension, similar to the mood of "Otherworldly Tales Logo"?


In [11]:
result2 = search_system.search_music(query=rewrite_query, top_k=200,use_top_n_context=1)

[2025-06-17T00:24:50Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


In [12]:
df_recommendations2 = pd.DataFrame(result2["final_results"])
df_recommendations2

,song_name,artist,mood,video_theme,genre,instrument,other_tags,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path,combined_info
0,The Truth Is Close,Artlist Musical Logos,"Serious, Dramatic, Dark","Technology, Science, Intros & Logos",Electronic,"Synth, Pads",,A dark and suspenseful piece of electronic mus...,0.848915,0.618877,0.947502,"audio,text",0.653127,music/Artlist Musical Logos - The Truth Is Clo...,"Moods: Serious, Dramatic, Dark. Video Themes: ..."
1,Wonders of the Universe,Artlist Musical Logos,"Dramatic, Tense, Mysterious","Documentary, Shorts, Intros & Logos",Cinematic,"Strings, Bells, Orchestra, Brass",,A mysterious and suspenseful orchestral track ...,0.921229,0.872366,0.942170,"audio,text",0.597490,music/Artlist Musical Logos - Wonders of the U...,"Moods: Dramatic, Tense, Mysterious. Video Them..."
2,Into the Edge,Pandocrator,"atmosphere, atmospheric, dark, dramatic, epic,...","action, cinema, film, hollywood, intro, movie,...","ambient, cinematic, drama",drums,"modern, soundtrack","A powerful, dark and edgy trailer track with a...",0.898948,0.760193,0.958415,"audio,text",0.563438,music/Pandocrator - Into the Edge.wav,"Moods: atmosphere, atmospheric, dark, dramatic..."
3,Emotional Piano and Percussion 1,Artlist Musical Logos,"Uplifting, Serious, Dramatic, Hopeful, Epic","Landscape, Travel, Shorts, Intros & Logos",Cinematic,"Piano, Strings, Percussion, Synth",,A cinematic and suspenseful track with a stron...,0.848377,0.749447,0.890776,"audio,text",0.400632,music/Artlist Musical Logos - Emotional Piano ...,"Moods: Uplifting, Serious, Dramatic, Hopeful, ..."
4,Action Opening,Artlist Musical Logos,"Dramatic, Angry, Tense, Dark","Trailer, Intros & Logos",Cinematic,"Electric Guitar, Synth, Electronic Drums, Pads",,The music is epic and cinematic with a driving...,0.854688,0.758540,0.895894,"audio,text",0.331012,music/Artlist Musical Logos - Action Opening.mp3,"Moods: Dramatic, Angry, Tense, Dark. Video The..."
5,Cinematic Epic Trailer Intro,PaBlikMM01,"elegant, epic, energetic, driven, boom, rising...","logo, opener, trailer, intro, animation, comme...","electronic, cinematic","synth, riser","new, soundtrack","This track features a big cinematic orchestra,...",0.868302,0.761365,0.914132,"audio,text",0.285870,music/PaBlikMM01 - Cinematic Epic Trailer Intr...,"Moods: elegant, epic, energetic, driven, boom,..."
6,Cyber Warfare,Artlist Musical Logos,"Serious, Dramatic, Tense, Dark","Drone Shots, Landscape, Slow Motion, Shorts, I...","Cinematic, Fantasy","Strings, Bells, Orchestra",,"A dark and intense track with heavy guitars, b...",0.918420,0.728067,1.000000,"audio,text",0.182717,music/Artlist Musical Logos - Cyber Warfare.mp3,"Moods: Serious, Dramatic, Tense, Dark. Video T..."
7,Greatest Epic Opener,ArchitectSound,"apocalyptic, dark, dramatic, emotional, energe...","action, adventure, battle, blockbuster, hollyw...","cinematic, cyberpunk, fantasy, orchestral",,soundtrack,This is a powerful cinematic track with a dram...,0.885652,0.795306,0.924372,"audio,text",0.133754,music/ArchitectSound - Greatest Epic Opener.mp3,"Moods: apocalyptic, dark, dramatic, emotional,..."
8,Otherworldly Tales Logo,Artlist Musical Logos,"Love, Dramatic, Dark, Mysterious","Time-Lapse, Documentary, Landscape, Slow Motio...","Ambient, Cinematic","Piano, Strings, Backing Vocals, Pads",,"A dark, suspenseful, and dramatic track featur...",0.858021,0.820532,0.874088,"audio,text",0.096706,music/Artlist Musical Logos - Otherworldly Tal...,"Moods: Love, Dramatic, Dark, Mysterious. Video..."
9,Emotional String Section Intro,Artlist Musical Logos,"Serious, Dramatic, Hopeful, Epic, Mysterious","Drone Shots, Landscape, Slow Motion, Shorts, I...","Cinematic, Fantasy","Strings, Bells, Orchestra",,This is a dramatic and emotional orchestral th...,0.839508,0.719215,0.891061,"audio,text",0.085099,music/Artlist Musical Logos - Emotional String...,"Moods: Serious, Dramatic, Hopeful, Epic, Myste..."


In [13]:
memory.save_context(
        {"input": query2},
        {"output": result2["explanation"]}
    )

In [14]:
memory_data = memory.load_memory_variables({})
print(memory_data["history"])

[HumanMessage(content='Someone is slowly exploring a dark, abandoned castle.', additional_kwargs={}, response_metadata={}), AIMessage(content='"Otherworldly Tales Logo" is an ideal match for exploring a dark, abandoned castle. Its dark, mysterious, and dramatic mood aligns perfectly with the eerie and suspenseful atmosphere one might experience in such a setting. The use of piano, strings, and synths creates a tense and unsettling ambiance, enhancing the sense of exploration and discovery in a haunted environment. This ambient, cinematic track is well-suited for capturing the slow, suspenseful journey through the castle\'s shadowy corridors. Its suspenseful nature complements the user\'s need for a soundtrack that evokes mystery and intrigue.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I need a soundtrack that creates a sense of tension.', additional_kwargs={}, response_metadata={}), AIMessage(content='"The Truth Is Close" by Artlist Musical Logos is an excelle

In [15]:
round_1 = df_recommendations.head(1)['audio_path'].iloc[0]
round_2 = df_recommendations2.head(1)['audio_path'].iloc[0]
print(round_1, round_2)

music/Artlist Musical Logos - Otherworldly Tales Logo.mp3 music/Artlist Musical Logos - The Truth Is Close.mp3


In [18]:
music_round_1 = client.files.upload(file=round_1)
music_round_2 = client.files.upload(file=round_2)

In [19]:
PAIR_PROMPT = SYS_PAIR_PROMPT
PAIR_PROMPT += f"\n\nHere is the pairing rule:\n{OVERALL_SCORE_PAIRING_PROMPT}"
print(PAIR_PROMPT)


You are a loyal judge, your task is to choose the better one from two responses on the given task. You will be given a task, including the input and the two responses. The pairing rule will also be given, you need to choose with your careful consideration. Judge task require multi-modal inputs, you should use your visual and auditory senses to judge. You should entirely understand, see or hear the task and the response, base on the given information, you should think of your choosing reasons in the each rubric’s "comment" step by step first, and then you are required to give a choice in "choice" base on the rule.
**Choosing Rule:**
Reasoning in detail before you determine the choice, then give your choice from [0,1,2], 0 means the first response is better, 1 means the two responses are equally good, 2 means the second response is better.


Here is the pairing rule:

You are going to choose base on the overall quality of the reponse's performance on the given task.
Overall Quality Defi

In [22]:
top1_pair = model_pair_content('video', input, music_round_2, music_round_1)
top1_pair

['Here is the query of video to music retrieval task:\nThis is a/an video . Please evaluate the following two background music based on this video. Which one is more suitable?',
 File(name='files/8vr8dtqugu65', display_name=None, mime_type='video/mp4', size_bytes=3524681, create_time=datetime.datetime(2025, 6, 16, 22, 28, 24, 911973, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 18, 22, 28, 24, 872412, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 6, 16, 22, 28, 24, 911973, tzinfo=TzInfo(UTC)), sha256_hash='NThiOTQ0MWNkZDg0YTVjNmQxMmY3OGNmZTlmYmU0MzQyYjliN2E1YmE2NGFmYmNhYWRkNDMwMGRmYjI0N2M0Zg==', uri='https://generativelanguage.googleapis.com/v1beta/files/8vr8dtqugu65', download_uri=None, state=<FileState.PROCESSING: 'PROCESSING'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None),
 '\nHere is the first reponse:\n',
 File(name='files/yyhj5urapdg8', display_name=None, mime_type='audio/mpeg', size_bytes=748739, create_time=datetime.da

In [23]:
votes, comments = run_vote(client, GEMINI_2_5_PRO, top1_pair, PAIR_PROMPT, 5)
final_decision, vote_summary = majority_vote(votes)

print(f"最終結果：模型 {final_decision} 勝出")
print("投票統計：", vote_summary)
print("評語範例：")
for c in comments:
    print("-", c)

最終結果：模型 2 勝出
投票統計： {'2': 5}
評語範例：
- The video depicts an energetic protest scene with people shouting and raising their fists. Response 2, with its driving rock music, better captures the raw energy, passion, and rebellious spirit of a protest. Response 1, while having an epic and powerful feel, sounds more like a news intro or a cinematic score for a grand, resolved event, which doesn't quite match the immediate and active energy of the protest shown.
- The video depicts a passionate scene, likely a protest or rally, with a woman using a megaphone and people in the background raising their hands. The first music option is orchestral and sounds more like a news theme or a dramatic announcement, which doesn't quite match the raw energy of the scene. The second music option is rock music with a strong, driving beat and energetic guitars. This style of music is much more fitting for a protest, conveying a sense of empowerment, urgency, and passion that aligns well with the visuals.
- The 

In [16]:
query3 = '''Preferably something with bell sounds to enhance the eerie atmosphere."'''
rewrite_query2 = search_system.rewrite_query_from_history(query3, memory) # rewrite

In [17]:
print(rewrite_query2)

I'm looking for a soundtrack that creates a sense of tension with an eerie atmosphere, preferably featuring bell sounds.


In [18]:
result3 = search_system.search_music(query=rewrite_query2, top_k=200,use_top_n_context=1)
df_recommendations3 = pd.DataFrame(result3["final_results"])
df_recommendations3

[2025-06-17T00:25:28Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,mood,video_theme,genre,instrument,other_tags,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path,combined_info
0,Space Race - Short Version,Dan Ayalon,"Serious, Dramatic, Sad",Science,"Ambient, Cinematic","Acoustic Guitar, Bells, Synth",,A mysterious and tense music track featuring a...,0.769271,0.534994,0.869676,"audio,text",0.940898,music/Dan Ayalon - Space Race - Short Version.mp3,"Moods: Serious, Dramatic, Sad. Video Themes: S..."
1,Horror Trailer,OhmLab,"suspense, scary, freaky, haunting, chilling, a...","zombie, trailer, commercial, movie, thriller, ...","horror, cinematic",,"soundscape, soundtrack",Dark and eerie atmospheric soundscape. Scary h...,0.902828,0.676093,1.000000,"audio,text",0.940681,music/OhmLab - Horror Trailer.mp3,"Moods: suspense, scary, freaky, haunting, chil..."
2,Into the Edge,Pandocrator,"atmosphere, atmospheric, dark, dramatic, epic,...","action, cinema, film, hollywood, intro, movie,...","ambient, cinematic, drama",drums,"modern, soundtrack","A powerful, dark and edgy trailer track with a...",0.702173,0.642801,0.727618,"audio,text",0.913677,music/Pandocrator - Into the Edge.wav,"Moods: atmosphere, atmospheric, dark, dramatic..."
3,Wonders of the Universe,Artlist Musical Logos,"Dramatic, Tense, Mysterious","Documentary, Shorts, Intros & Logos",Cinematic,"Strings, Bells, Orchestra, Brass",,A mysterious and suspenseful orchestral track ...,0.769539,0.578517,0.851406,"audio,text",0.852935,music/Artlist Musical Logos - Wonders of the U...,"Moods: Dramatic, Tense, Mysterious. Video Them..."
4,Talk - Short Version,Alon Ohana,"Serious, Dramatic, Tense, Scary, Dark","Time-Lapse, Drone Shots, Slow Motion, Trailer","Ambient, Cinematic, Classical","Strings, Bells, Synth, Backing Vocals, Keys, Pads",,Mysterious and dark cinematic track with a bea...,0.782646,0.658223,0.835970,"audio,text",0.678114,music/Alon Ohana - Talk - Short Version.mp3,"Moods: Serious, Dramatic, Tense, Scary, Dark. ..."
5,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,"The music is mysterious and dramatic, featurin...",0.701843,0.751630,0.680505,"audio,text",0.562657,music/Swirling Ship - Fixed - Short Version B.mp3,"Moods: Serious, Dramatic, Scary, Dark. Video T..."
6,Black Magic Drama Tension,FlossieWood,"dark, magic, magical, mysterious, mystery, sec...","black magic, film, spy, spying, thriller",drama,"bells, piano, triangle",,"A piano and orchestra play a suspenseful, acti...",0.783945,0.520693,0.896768,"audio,text",0.492142,music/FlossieWood - Black Magic Drama Tension.mp3,"Moods: dark, magic, magical, mysterious, myste..."
7,Cyber Warfare,Artlist Musical Logos,"Serious, Dramatic, Tense, Dark","Drone Shots, Landscape, Slow Motion, Shorts, I...","Cinematic, Fantasy","Strings, Bells, Orchestra",,"A dark and intense track with heavy guitars, b...",0.735911,0.534702,0.822143,"audio,text",0.272315,music/Artlist Musical Logos - Cyber Warfare.mp3,"Moods: Serious, Dramatic, Tense, Dark. Video T..."
8,Creeping Prank,Nuclearmetal,"crazy, creepy, dark, enchanted, fear, fun, ins...","cartoon, comedy, halloween, intro, kids, movie...",horror,"bells, orchestra, strings",loop,"A short, quirky and funny track featuring gloc...",0.754698,0.672566,0.789897,"audio,text",0.248352,music/Nuclearmetal - Creeping Prank.mp3,"Moods: crazy, creepy, dark, enchanted, fear, f..."
9,Mysterious Musical Ident 2,Artlist Musical Logos,"Dramatic, Sad, Playful, Dark, Mysterious","Time-Lapse, Documentary, Landscape, Nature, Sl...",Cinematic,"Piano, Bells, Pads",,A beautiful and calming orchestral piece that ...,0.687735,0.734118,0.667857,"audio,text",0.128525,music/Artlist Musical Logos - Mysterious Music...,"Moods: Dramatic, Sad, Playful, Dark, Mysteriou..."


In [19]:
top_k = 5

round1 = df_recommendations['song_name'].head(top_k).tolist()
round2 = df_recommendations2['song_name'].head(top_k).tolist()
round3 = df_recommendations3['song_name'].head(top_k).tolist()

print(round1)
print(round2)
print(round3)

['Otherworldly Tales Logo', 'Talk - Short Version', 'Cyber Warfare', 'Fixed - Short Version B', 'Black Magic Drama Tension']
['The Truth Is Close', 'Wonders of the Universe', 'Into the Edge', 'Emotional Piano and Percussion 1', 'Action Opening']
['Space Race - Short Version', 'Horror Trailer', 'Into the Edge', 'Wonders of the Universe', 'Talk - Short Version']


In [20]:
evaluator = RankingEvaluator(top_k)

gt = Q7_GT
song_pool = df_recommendations["song_name"].tolist()
results = [evaluator.evaluate_random_baseline(gt, song_pool, n=10)] #Random 10 times

methods = ['Round1', 'Round2', 'Round3']
recommendations = [round1, round2, round3]

for method, rec in zip(methods, recommendations):
    results.append(evaluator.evaluate(gt, rec, method))

df = pd.DataFrame(results)
df

,Method,Precision@5,Recall@5,nDCG@5,MAP
0,Baseline (Random),0.16,0.133333,0.130682,0.045278
1,Round1,0.40,0.333333,0.345191,0.150000
2,Round2,0.20,0.166667,0.213986,0.083333
3,Round3,0.60,0.500000,0.616434,0.350000


In [5]:
print(results["choice"])

openai


In [6]:
print(results["retrieval_context"])

1. Ziv Moran - Promised - Short Version B
Information: Moods: Uplifting, Powerful, Hopeful, Groovy, Exciting. Video Themes: Road Trip, Lifestyle, Urban, Medical, Landscape, Travel. Instruments: Acoustic Guitar, Piano, Acoustic Drums, Bells, Backing Vocals. Genres: Acoustic, Pop, Folk. Other tags: . Description: A driving, upbeat, acoustic guitar-based instrumental with a driving beat and a sense of movement and progress. The mood is positive and uplifting, with a sense of accomplishment and satisfaction. The main instruments are acoustic guitar, piano, and drums. The genre/style is pop, folk, and instrumental. Suitable uses for this track include background music for corporate videos, presentations, and commercials, as well as in film and television soundtracks.




In [7]:
print(results["explanation"])

The user's need for a tool to clear browsing history does not relate to music preferences, making it challenging to establish a connection with the recommended song. "Promised - Short Version B" by Ziv Moran is an uplifting and powerful acoustic track, characterized by its driving beat and sense of progress. While the song's positive and hopeful mood might metaphorically align with a sense of digital 'cleanliness' or 'fresh start,' it doesn't directly address the user's specific request. Therefore, no strong musical connection can be reasonably inferred.


In [8]:
print(results["explanation_prompt"])

You are a professional music recommendation assistant.
        Based on the following user need and the details of recommended songs, please generate a natural, clear, and engaging explanation.
        
        User needs: I’m looking for a tool to clear my browsing history.

        Recommended songs:
        1. Ziv Moran - Promised - Short Version B
Information: Moods: Uplifting, Powerful, Hopeful, Groovy, Exciting. Video Themes: Road Trip, Lifestyle, Urban, Medical, Landscape, Travel. Instruments: Acoustic Guitar, Piano, Acoustic Drums, Bells, Backing Vocals. Genres: Acoustic, Pop, Folk. Other tags: . Description: A driving, upbeat, acoustic guitar-based instrumental with a driving beat and a sense of movement and progress. The mood is positive and uplifting, with a sense of accomplishment and satisfaction. The main instruments are acoustic guitar, piano, and drums. The genre/style is pop, folk, and instrumental. Suitable uses for this track include background music for corporate vi

In [9]:
judge_llm = CustomClaudeSonnet()

In [10]:
query = query 
generated_output = results["explanation"]
context_docs = [results["retrieval_context"]]

results = evaluate_all_metrics_with_usefulness(query, generated_output, context_docs, model = judge_llm)
output = {}
for test_result in results.test_results:
    for metric_result in test_result.metrics_data:
        output[metric_result.name] = {
            "score": metric_result.score,
            "reason": getattr(metric_result, "reason", "N/A"),
            "pass": metric_result.success
        }
output

✨ You're running DeepEval's latest Faithfulness Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Usefulness (GEval) Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

Evaluating 1 test case(s) in parallel: |██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|100% (1/1) [Time Taken: 00:24, 24.79s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: Claude-3.5-Sonnet, reason: The score is 1.00 because the response shows perfect faithfulness! There are no contradictions between the actual output and the retrieval context, indicating the model did an excellent job staying true to the source material., error: None)
  - ❌ Answer Relevancy (score: 0.6666666666666666, threshold: 0.7, strict: False, evaluation model: Claude-3.5-Sonnet, reason: The score is 0.67 because while the response did address browser history clearing tools, it contained unnecessary and off-topic information about music tracks and song characteristics that had nothing to do with the user's request for browser history clearing assistance. The score still remains above average since the core request was addressed, but the irrelevant musical details prevented it from scoring higher., error: None)
  - ✅ Usefulness (GEval) (score: 0.9, threshold: 0.7, strict: False, evalu

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

{'Faithfulness': {'score': 1.0,
  'reason': 'The score is 1.00 because the response shows perfect faithfulness! There are no contradictions between the actual output and the retrieval context, indicating the model did an excellent job staying true to the source material.',
  'pass': True},
 'Answer Relevancy': {'score': 0.6666666666666666,
  'reason': "The score is 0.67 because while the response did address browser history clearing tools, it contained unnecessary and off-topic information about music tracks and song characteristics that had nothing to do with the user's request for browser history clearing assistance. The score still remains above average since the core request was addressed, but the irrelevant musical details prevented it from scoring higher.",
  'pass': False},
 'Usefulness (GEval)': {'score': 0.9,
  'reason': "The explanation acknowledges the mismatch between the browsing history request and music recommendation, shows honesty by clearly stating 'no strong musical 